# Phase 3 (Rhetorical Roles: CL + IT) — Run on Phase-1 **Decoder-only model 03** (ALiBi + Multi-Head Latent Attention), with **rsLoRA**

Config-driven, like the other Phase-3 notebooks: edit **Cell 2** only. This is the optional fifth run of your plan: the **full-size** Dec-03 (13 blocks, 53.9M parameters), unmasked through symmetric ALiBi, adapted with **rank-stabilised LoRA (rsLoRA)** at a high rank. Data handling, windows, metrics, bias tuning and the submission writer are the same as in the other Phase-3 notebooks, so all runs are comparable. It also gives the KD student (a 7-block cut of this same model) its full-size sibling.

### The task in one paragraph
Every document is an ordered list of sentences; predict **one of 13 rhetorical-role labels per sentence** (order read from `label_vocab.json`). Metric: **macro-F1**. Model = Phase-1 backbone + **mean-pool over the tokens of each sentence** + `Linear(512 → 13)` (the sentence-level Module 2.4 recipe). Context: windows of consecutive sentences (`CONTEXT_MODE="window"`) or one sentence per sequence (`"sentence"`, the no-context control).

### What this architecture offers
| | Decoder 01 | Decoder 02 | **Decoder 03 (this notebook)** |
|---|---|---|---|
| positions | none (NoPE) | RoPE | **ALiBi**: a fixed bias `−m_h·(i−j)` on the attention logits, no position parameters |
| attention | MHA | local/global + GQA | **MLA**: K and V of every head are rebuilt from a 128-dim latent `c_t` |
| layers | 12 | 12 | **13** |
| can it be unmasked? | no | yes | **yes**: the distance lives in the *bias*, so the symmetric version `−m_h·|i−j|` is well defined |

### rsLoRA (Kalajdzievski 2023) — why the scaling factor matters
LoRA adds `γ · B·A·x` to a frozen layer, with `A ∈ R^{r×d_in}` (Kaiming-uniform, entries of variance ≈ 1/(3·d_in)) and `B = 0` at the start. Standard LoRA uses `γ = α/r`. Take **one SGD step** on `B` with a linear loss `L = g·h`: `∇_B L = γ·g·(A x)ᵀ`, so `B₁ = −η·γ·g·(A x)ᵀ` and the change of the layer output is

`Δh = γ·B₁·A·x = −η·γ²·‖A x‖²·g`,  with  `‖A x‖² = Σᵢ (aᵢ·x)² ≈ r·‖x‖²/(3·d_in)`  (it grows **linearly in r**).

So `|Δh| ∝ γ²·r`. With `γ = α/r` this is `∝ α²/r`: **the larger the rank, the smaller the update**, which is why plain LoRA at high rank learns no better than at low rank (the extra capacity is never used). The unique choice that keeps `γ²·r` constant, i.e. makes learning *stable and rank-independent*, is **`γ = α/√r`**. That is the whole of rsLoRA: a single change of the scaling constant, no extra parameters. The notebook prints a numerical check of this (`SCALING_DEMO`, Cell 17).

Which adapters: every `Linear` of the model that is named in `ADAPTER_TARGETS` (MLA: `q_proj`, `kv_down`, `k_up`, `v_up`, `o_proj`; SwiGLU: `gate_proj`, `up_proj`, `down_proj`). At the default rank 64: about 8.1M trainable parameters (15 % of the model) against 2.0M for LoRA at rank 16.

### `METHOD`
| `METHOD` | What is trained |
|---|---|
| `"rslora"` | rsLoRA matrices (`γ = α/√r`, default r = 64, α = 16) + head (**default**) |
| `"lora"` | standard LoRA (`γ = α/r`, default r = 16, α = 32) + head |
| `"partial"` | the top `PARTIAL_K` blocks + final norm + head (cheap middle ground, run with `ATTN_MODE="bidir_top"`) |
| `"full"` | everything (full-fine-tuning reference) |
| `"norm_only"` | all RMSNorm gains (incl. MLA's latent norm) + head |

### `ATTN_MODE`
`"bidir"` (default): all 13 blocks use symmetric ALiBi and no causal mask, so each sentence sees its whole window. `"bidir_top"`: the lower blocks stay causal (exactly the Phase-1 function), the top `BIDIR_TOP_K` blocks are unmasked; the natural partner of `partial`, where the frozen lower blocks must not see an attention pattern they were never trained for. `"echo"` / `"causal"`: as in the Dec-01 notebook.

### Class imbalance, domains, outputs
Class-weighted cross-entropy (`sqrt_inv`); a per-class logit bias tuned on dev **and kept only if a 2-fold document-level check shows a held-out gain**; CL and IT trained jointly with a text tag; selection score = mean of the per-domain macro-F1; one `prediction.json` with all CL and IT test documents. Outputs go to `WORK_DIR/runs/<RUN_NAME>/`: `run_log.json`, `rr_*.pt`, `submission/submission.zip`, optionally `teacher_softlabels_*.pt`.

In [ ]:
# Cell 0 — Hugging Face login (token from Kaggle secrets), same as the Phase-1 notebooks
import os
try:
    from huggingface_hub import login
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF")
    os.environ["HF_TOKEN"] = _tok
    login(_tok)
    print("HF login ok")
except Exception as e:
    print("HF secret not available:", e)

In [ ]:
# Cell 1 — Imports, paths, device, precision (same conventions as the Phase-1 notebooks)
import os, sys, json, math, time, random, gc, zipfile, types, re, pickle
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import Optional, Dict, List, Tuple, Any
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("./work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_DIR = Path("/kaggle/input/datasets/ascyrax/capstone-tokenizer-large") if IS_KAGGLE else Path("./input")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    cap = torch.cuda.get_device_capability(0)
    AMP_DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16     # T4/P100 -> fp16 + GradScaler
    USE_AMP = True
    print("GPU:", torch.cuda.get_device_name(0), "| capability", cap)
else:
    AMP_DTYPE, USE_AMP = torch.float32, False
USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16
print(f"device={device} | AMP dtype={AMP_DTYPE} | grad scaler={USE_SCALER} | torch {torch.__version__}")

SEED = 1337
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if device.type == "cuda": torch.cuda.manual_seed_all(s)
seed_everything(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [ ]:
# Cell 2 — RUN CONFIG  (the only cell you normally edit)

RUN_NAME   = "p3_dec03_rslora"
METHOD     = "rslora"         # "rslora" | "lora" | "partial" | "full" | "norm_only"
SMOKE_TEST = False            # True: ~250 steps + a few dev docs, dev check only, NO submission written
ATTN_MODE  = "bidir"          # "bidir" | "bidir_top" | "echo" | "causal"   (see the first markdown cell)
PARTIAL_K   = 4               # METHOD="partial": train the top K blocks (0 = final norm + head only, a linear probe)
BIDIR_TOP_K = 4               # ATTN_MODE="bidir_top": the top K blocks become bidirectional (keep equal to PARTIAL_K for partial)
SCALING_DEMO = True           # print the one-step numerical check of the LoRA vs rsLoRA scaling (Cell 17), takes a second

# ---- Phase-1 checkpoint (written by caps_decoder_only_architecture_03.ipynb: runs/<ARCH>/best.pt) ---------------------------------
CKPT_PATH = Path("/kaggle/input/datasets/ascyrax/caps-dec-03-best/best.pt") if IS_KAGGLE else Path("./input/best.pt")

# ---- Data ---------------------------------------------------------------------------------
HF_REPO = "Exploration-Lab/CS-781-Capstone"      # files are Phase_3_RR/{Train,Test}/...
LOCAL_DATA_DIR = None                             # or a folder that contains Phase_3_RR/ (checked first)
DOMAINS    = ["CL", "IT"]                         # both domains are trained jointly; use ["CL"] or ["IT"] for a single-domain run
DOMAIN_TAG = True                                 # prepend a short text tag naming the domain
FINAL_FIT  = False                                # True: train on train+dev, NO validation (use after the dev run, EPOCHS = best epoch)

# ---- Context (the sentence-level vs document-level context ablation) ----------------------------
CONTEXT_MODE   = "window"                         # "window" (many sentences per sequence) | "sentence" (one sentence per sequence, no context)
WINDOW_TOKENS  = 800                              # TOTAL tokens per sequence (<= 1024); with ATTN_MODE="echo" each copy gets (WINDOW_TOKENS-1)//2
WINDOW_OVERLAP = 0.5                              # fraction of a window that the next window repeats
MAX_SENT_TOK   = 160                              # each sentence is truncated to this many tokens
EDGE_WEIGHT    = True                             # at inference weigh a sentence's prediction by its distance from the window edge

# ---- Training -----------------------------------------------------------------------------
EPOCHS       = 6
BATCH_WINDOW = 8                                  # windows per micro-batch (CONTEXT_MODE="window")
BATCH_SENT   = 64                                 # sentences per micro-batch (CONTEXT_MODE="sentence")
ACCUM        = 2                                  # gradient accumulation
WARMUP_FRAC  = 0.06
GRAD_CLIP    = 1.0
HP = {                                            # per-method hyper-parameters (starting points, tune on dev)
    "rslora":    dict(lr=5e-4, head_lr=1e-3, wd=0.0, r=64, alpha=16, dropout=0.05),     # gamma = alpha / sqrt(r) = 2.0
    "lora":      dict(lr=5e-4, head_lr=1e-3, wd=0.0, r=16, alpha=32, dropout=0.05),     # gamma = alpha / r      = 2.0
    "partial":   dict(lr=2e-4, head_lr=1e-3, wd=0.01),
    "full":      dict(lr=1e-4, head_lr=1e-3, wd=0.01),
    "norm_only": dict(lr=1e-3, head_lr=1e-3, wd=0.0),
}
# every Linear name in this architecture: MHA/GQA-style (q,k,v,o), MLA (q, kv_down, k_up, v_up, o) and the SwiGLU FFN
ADAPTER_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "kv_down", "k_up", "v_up", "gate_proj", "up_proj", "down_proj")
HEAD_DROPOUT    = 0.1
CLASS_WEIGHT    = "sqrt_inv"                      # "sqrt_inv" | "none"
MAX_CLASS_W     = 10.0
LABEL_SMOOTHING = 0.0

# ---- Per-class logit bias tuned on dev (macro-F1 post-processing) -------------------------------
BIAS_TUNE     = True
BIAS_MIN_GAIN = 0.005                             # keep the tuned bias only if a 2-fold document-level check on dev gains this much macro-F1
BIAS_FINAL    = None                              # used when FINAL_FIT=True: paste the bias list printed by the dev run (or None)

# ---- Smoke test -----------------------------------------------------------------------------
SMOKE_STEPS, SMOKE_VAL_DOCS = 250, 10

# ---- Knowledge distillation (run #4): let this run serve as the TEACHER (it is larger than the 7-block student) -----------------
EXPORT_SOFT_LABELS = False        # True (full run): save per-sentence log-probabilities of the train and dev documents

# ---- derived ----------------------------------------------------------------------------------
assert METHOD in HP and ATTN_MODE in ("causal", "echo", "bidir", "bidir_top") and CONTEXT_MODE in ("window", "sentence")
assert 0.0 <= WINDOW_OVERLAP < 1.0 and WINDOW_TOKENS <= 1024, "sequences longer than 1024 exceed the Phase-1 training context"
SEQ_BUDGET = (WINDOW_TOKENS - 1) // 2 if ATTN_MODE == "echo" else WINDOW_TOKENS     # tokens available for ONE copy of a window
assert SEQ_BUDGET >= MAX_SENT_TOK + 32, "a window copy must hold at least one (truncated) sentence plus the tags (~25 tokens)"
BATCH = BATCH_WINDOW if CONTEXT_MODE == "window" else BATCH_SENT
RUN_DIR = WORK_DIR / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN {RUN_NAME} | METHOD={METHOD} | ATTN_MODE={ATTN_MODE} | CONTEXT_MODE={CONTEXT_MODE} (window={WINDOW_TOKENS}, per-copy budget={SEQ_BUDGET}, "
      f"overlap={WINDOW_OVERLAP}) | domains={DOMAINS} | final_fit={FINAL_FIT} | smoke={SMOKE_TEST}")

In [ ]:
# Cell 3 — The shared byte-level BPE tokenizer (same as every Phase-1 model)
if str(TOKENIZER_DIR) not in sys.path:
    sys.path.append(str(TOKENIZER_DIR))
from bpe_tokenizer import BPETokenizer, load_tokenizer

tokenizer = load_tokenizer(str(TOKENIZER_DIR))
VOCAB_SIZE = len(tokenizer.id_to_bytes)
BOS_ID, EOS_ID, PAD_ID, UNK_ID = tokenizer.bos_id, tokenizer.eos_id, tokenizer.pad_id, tokenizer.unk_id
print("vocab_size:", VOCAB_SIZE, "| BOS", BOS_ID, "EOS", EOS_ID, "PAD", PAD_ID)

# Sentences are tokenised exactly as given (no clean_text / normalisation).
_s = "Hon'ble Mr. Justice R. K. Gauba — § 302 IPC, ₹ 5,000 “quoted”."
_ids, _offs = tokenizer.encode(_s, return_offsets=True)
assert tokenizer.decode(_ids) == _s, "tokenizer round-trip failed"
assert all(0 <= a <= b <= len(_s) for a, b in _offs) and _offs[0][0] == 0 and _offs[-1][1] == len(_s)
print("offset sanity ok |", len(_ids), "tokens for", len(_s), "chars")

# Guard against O(L^2) BPE on pathological chunks (long runs of dashes, dots, underscores, spaces) -- the same guard the Phase-1
# Decoder-03 notebook uses. Pre-tokens longer than 1000 bytes are BPE-encoded in 1000-byte pieces. Normal text never has such
# chunks, so its tokenisation (and every character offset) is unchanged; byte-level BPE decodes the pieces to the exact same bytes.
_MAX_PRETOK_BYTES = 1000
_orig_encode_pretok = tokenizer._encode_pretok_bytes
def _encode_pretok_bytes_capped(bs):
    if len(bs) <= _MAX_PRETOK_BYTES:
        return _orig_encode_pretok(bs)
    out = []
    for i in range(0, len(bs), _MAX_PRETOK_BYTES):
        out.extend(_orig_encode_pretok(bs[i:i + _MAX_PRETOK_BYTES]))
    return out
tokenizer._encode_pretok_bytes = _encode_pretok_bytes_capped
_t0 = time.time(); _ids = tokenizer.encode("-" * 200_000)
assert tokenizer.decode(_ids) == "-" * 200_000
print(f"long-run guard ok: 200k-char run encoded in {time.time() - _t0:.1f}s")

## Phase-1 model code — copied **verbatim** from `caps_decoder_only_architecture_03.ipynb` (Cells 4-11)
Do not edit: the checkpoint only loads correctly if these classes are identical.

In [ ]:
# Cell 4 — Model configuration
#
# One config drives three architectures (all registered, all testable):
#   preset_alibi_mla()  -> the NEW architecture (ALiBi positions + Multi-Head Latent Attention)
#   preset_alibi_mha()  -> ablation: ALiBi with ordinary MHA (isolates the ALiBi effect)
#   preset_rope_mha()   -> corrected baseline: RoPE + MHA (model 01 accidentally had NO positions)

@dataclass
class ModelConfig:
    # --- shape ---
    vocab_size: int = 32000
    n_layer: int = 13
    d_model: int = 512
    n_head: int = 8
    n_kv_head: int = 8                # only used by mixer_type="attention" (GQA if < n_head)
    head_dim: int = 64
    ff_mult: float = 8/3              # SwiGLU hidden = round64(ff_mult * d_model) = 1408
    d_ff: Optional[int] = None
    max_seq_len: int = 1024           # TRAINING context. ALiBi itself has no length limit.

    # --- components ---
    norm_type: str = "rmsnorm"        # "rmsnorm" | "layernorm"
    norm_eps: float = 1e-5
    pos_type: str = "alibi"           # "alibi" | "rope" | "learned" | "none"
    rope_theta: float = 10000.0
    mixer_type: str = "mla"           # "attention" (MHA/GQA) | "mla"
    kv_latent_dim: int = 128          # d_c: MLA caches only this many numbers per token per layer
    mla_latent_norm: bool = True      # RMSNorm on the latent c_t (as in DeepSeek-V2)
    mlp_type: str = "swiglu"          # "swiglu" | "gelu"
    channel_mixer_type: str = "mlp"

    # --- misc ---
    bias: bool = False
    attn_dropout: float = 0.0
    resid_dropout: float = 0.0
    embd_dropout: float = 0.0
    tie_embeddings: bool = True
    init_std: float = 0.02
    use_sdpa: bool = True

    def __post_init__(self):
        assert self.d_model == self.n_head * self.head_dim, "d_model must equal n_head*head_dim"
        assert self.n_head % self.n_kv_head == 0, "n_head must be divisible by n_kv_head"
        assert self.pos_type in ("alibi", "rope", "learned", "none")
        assert self.mixer_type in ("attention", "mla")
        if self.mixer_type == "mla":
            # RoPE rotates q and k by a position-dependent matrix that sits BETWEEN W_Q and
            # W_UK, which breaks MLA's weight absorption (DeepSeek needs "decoupled RoPE").
            # ALiBi only adds a bias outside q.k, so it composes cleanly with MLA.
            assert self.pos_type != "rope", "MLA + RoPE needs decoupled RoPE; use ALiBi"
            assert not self.bias, "MLA absorption assumes bias-free up-projections"
        if self.d_ff is None:
            raw = int(self.ff_mult * self.d_model)
            self.d_ff = 64 * ((raw + 63) // 64)


def preset_alibi_mla(vocab_size=None, max_seq_len=1024):
    # MLA saves ~0.33M attention params per layer vs MHA (d_c=128). 13 layers instead of 12
    # reinvests that saving so the model stays parameter-matched to the 12-layer baseline.
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, n_layer=13, pos_type="alibi",
                       mixer_type="mla", kv_latent_dim=128, max_seq_len=max_seq_len)

def preset_alibi_mha(vocab_size=None, max_seq_len=1024):
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, n_layer=12, pos_type="alibi",
                       mixer_type="attention", max_seq_len=max_seq_len)

def preset_rope_mha(vocab_size=None, max_seq_len=1024):
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, n_layer=12, pos_type="rope",
                       mixer_type="attention", max_seq_len=max_seq_len)

PRESETS = {"alibi_mla": preset_alibi_mla, "alibi_mha": preset_alibi_mha, "rope_mha": preset_rope_mha}
print(json.dumps(asdict(preset_alibi_mla()), indent=2))

In [ ]:
# Cell 5 — Normalisation layers (hand-written)

class LayerNorm(nn.Module):
    """Classic LayerNorm: centre by the mean, scale by the std, learn gain g and bias b."""
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d))
        self.b = nn.Parameter(torch.zeros(d))
        self.eps = eps

    def forward(self, x):
        xf = x.float()
        mean = xf.mean(-1, keepdim=True)
        var = xf.var(-1, keepdim=True, unbiased=False)
        return (((xf - mean) / torch.sqrt(var + self.eps)).to(x.dtype)) * self.g + self.b


class RMSNorm(nn.Module):
    """RMSNorm: no mean-centring and no bias; x / rms(x) * g. Statistics computed in fp32."""
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() / rms).to(x.dtype) * self.g


def build_norm(cfg: ModelConfig, d: Optional[int] = None):
    d = d or cfg.d_model
    if cfg.norm_type == "layernorm":
        return LayerNorm(d, cfg.norm_eps)
    if cfg.norm_type == "rmsnorm":
        return RMSNorm(d, cfg.norm_eps)
    raise ValueError(cfg.norm_type)

In [ ]:
# Cell 6 — Positional mechanisms: ALiBi (new) and RoPE (fixed wiring)
#
# ALiBi (Press et al., 2022). No position embeddings at all. Each head h adds a fixed bias
#       s_ij = q_i.k_j / sqrt(d)  -  m_h * (i - j),        j <= i
# Inside the softmax this multiplies attention by exp(-m_h (i-j)): an exponential recency prior.
# Only the distance (i - j) matters: softmax ignores a per-row constant, and
# -m(i-j) = -m*i + m*j, where -m*i is constant along row i.
# Slopes are geometric, m_h = 2^(-8h/H), h = 1..H. For H = 8: 1/2, 1/4, ..., 1/256, so heads
# range from "looks only a few tokens back" to "looks hundreds of tokens back".

def alibi_slopes(n_head: int) -> List[float]:
    def pow2_slopes(n):
        start = 2.0 ** (-8.0 / n)
        return [start ** (i + 1) for i in range(n)]
    if math.log2(n_head).is_integer():
        return pow2_slopes(n_head)
    # Non-power-of-2 head counts: the paper's interpolation rule.
    c = 2 ** math.floor(math.log2(n_head))
    return pow2_slopes(c) + pow2_slopes(2 * c)[0::2][: n_head - c]


class ALiBi(nn.Module):
    def __init__(self, n_head: int):
        super().__init__()
        self.register_buffer("slopes", torch.tensor(alibi_slopes(n_head), dtype=torch.float32),
                             persistent=False)

    def bias(self, q_len: int, k_len: int, device) -> torch.Tensor:
        """[1, H, q_len, k_len] float32. Queries are the LAST q_len of the k_len positions
        (so it also works for KV-cached decoding). Includes the causal mask (-inf for j > i)."""
        q_pos = torch.arange(k_len - q_len, k_len, device=device)
        k_pos = torch.arange(k_len, device=device)
        dist = (q_pos[:, None] - k_pos[None, :]).float()          # i - j
        b = -self.slopes.to(device)[:, None, None] * dist          # [H, Tq, Tk]
        b = b.masked_fill(dist < 0, float("-inf"))
        return b.unsqueeze(0)


class RoPE(nn.Module):
    """Rotary embeddings. Table is built for 2x the training length so generation never overflows."""
    def __init__(self, head_dim: int, max_pos: int, theta: float = 10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
        freqs = torch.outer(torch.arange(max_pos).float(), inv_freq)   # [P, D/2]
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def tables(self, position_ids: torch.Tensor):
        """position_ids [B, T] -> (cos, sin) each [B, 1, T, D/2]."""
        return self.cos[position_ids].unsqueeze(1), self.sin[position_ids].unsqueeze(1)

    @staticmethod
    def apply(x, cos, sin):
        """x [B, H, T, D]; rotation done in fp32 for stability."""
        dt = x.dtype
        x = x.float()
        x1, x2 = x[..., 0::2], x[..., 1::2]
        out = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)
        return out.to(dt)

In [ ]:
# Cell 7 — Shared attention kernel + standard MHA/GQA token mixer
#
# The Backbone builds ONE additive mask per forward pass (`attn_bias`, shape [B or 1, H or 1, T, S])
# containing ALiBi + causal + padding, and passes it to every layer. Every token mixer then
# calls `attend`. When attn_bias is None we use SDPA's fast is_causal path.

def attend(q, k, v, attn_bias, is_causal, scale, dropout_p=0.0, use_sdpa=True):
    """q [B,H,T,Dq], k [B,H,S,Dq], v [B,H,S,Dv] -> [B,H,T,Dv]."""
    if attn_bias is not None:
        attn_bias = attn_bias.to(q.dtype)          # SDPA requires mask dtype == query dtype
        is_causal = False
    if use_sdpa:
        return F.scaled_dot_product_attention(q, k, v, attn_mask=attn_bias,
                                              dropout_p=dropout_p, is_causal=is_causal,
                                              scale=scale)
    # Manual reference path (used by the tests to cross-check SDPA).
    scores = (q.float() @ k.float().transpose(-2, -1)) * scale
    if is_causal:
        T, S = q.shape[-2], k.shape[-2]
        scores = scores.masked_fill(
            torch.triu(torch.ones(T, S, dtype=torch.bool, device=q.device), diagonal=S - T + 1),
            float("-inf"))
    if attn_bias is not None:
        scores = scores + attn_bias.float()
    p = F.softmax(scores, dim=-1)
    if dropout_p > 0:
        p = F.dropout(p, dropout_p)
    return (p @ v.float()).to(q.dtype)


class Attention(nn.Module):
    """Multi-head (n_kv_head == n_head) or grouped-query (n_kv_head < n_head) self-attention.
    Cache format: (k, v), each [B, n_kv_head, S, head_dim]."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.n_head, self.n_kv_head, self.head_dim = cfg.n_head, cfg.n_kv_head, cfg.head_dim
        self.scale = 1.0 / math.sqrt(cfg.head_dim)
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_head * cfg.head_dim, bias=cfg.bias)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.o_proj = nn.Linear(cfg.n_head * cfg.head_dim, cfg.d_model, bias=cfg.bias)
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)

    def _split(self, x, n):
        B, T, _ = x.shape
        return x.view(B, T, n, self.head_dim).transpose(1, 2)

    @staticmethod
    def cache_len(cache):
        return cache[0].shape[2]

    def cache_numbers_per_token(self):
        return 2 * self.n_kv_head * self.head_dim

    def forward(self, x, attn_bias=None, rope_cs=None, kv_cache=None):
        B, T, _ = x.shape
        q = self._split(self.q_proj(x), self.n_head)
        k = self._split(self.k_proj(x), self.n_kv_head)
        v = self._split(self.v_proj(x), self.n_kv_head)
        if rope_cs is not None:                       # FIX: RoPE is now actually applied
            cos, sin = rope_cs
            q, k = RoPE.apply(q, cos, sin), RoPE.apply(k, cos, sin)
        if kv_cache is not None:
            k = torch.cat([kv_cache[0], k], dim=2)
            v = torch.cat([kv_cache[1], v], dim=2)
        new_cache = (k, v)
        if self.n_kv_head != self.n_head:
            rep = self.n_head // self.n_kv_head
            k, v = k.repeat_interleave(rep, dim=1), v.repeat_interleave(rep, dim=1)
        out = attend(q, k, v, attn_bias, is_causal=(attn_bias is None and T > 1),
                     scale=self.scale,
                     dropout_p=self.cfg.attn_dropout if self.training else 0.0,
                     use_sdpa=self.cfg.use_sdpa)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_dropout(self.o_proj(out)), new_cache

In [ ]:
# Cell 8 — Multi-Head Latent Attention (MLA, DeepSeek-V2), adapted to ALiBi
#
#   c_t      = Norm(W_DKV x_t)          in R^{d_c}        (the ONLY thing we cache)
#   k_t^(h)  = W_UK^(h) c_t,   v_t^(h) = W_UV^(h) c_t    (every head gets its own K, V)
#   q_t^(h)  = W_Q^(h) x_t
#
# KV cache per token per layer: MHA stores 2*H*D_h = 1024 numbers; MLA stores d_c = 128 (8x less).
#
# MLA vs GQA: GQA *ties* heads (groups share identical K,V). MLA gives each head distinct K,V,
# all reconstructed from a shared rank-d_c subspace. With d_c = 2*n_kv*D_h, MLA can exactly
# express GQA (the up-projections just copy slices), so at equal cache size GQA is a special case.
#
# Weight absorption (inference trick, implemented below as `absorb=True`):
#   q_i^(h).k_j^(h) = (W_Q^(h) x_i)^T W_UK^(h) c_j = [W_UK^(h)T q_i^(h)] . c_j
# so we can attend directly against the cached latents without ever building K:
#   q_lat = W_UK^T q,  scores = q_lat . c,  out_lat = softmax(scores) c,  out^(h) = W_UV^(h) out_lat
# RoPE would insert a position-dependent rotation R_{i-j} between W_Q and W_UK and break this;
# ALiBi's bias lives outside the dot product, so absorption is unaffected.
# Training uses the materialised path (simpler, identical maths); a test checks both agree.

class MLAttention(nn.Module):
    """Cache format: c, [B, S, d_c]."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        H, Dh, dc = cfg.n_head, cfg.head_dim, cfg.kv_latent_dim
        self.n_head, self.head_dim, self.d_c = H, Dh, dc
        self.scale = 1.0 / math.sqrt(Dh)              # scores are still per-head q.k with dim D_h
        self.q_proj = nn.Linear(cfg.d_model, H * Dh, bias=False)
        self.kv_down = nn.Linear(cfg.d_model, dc, bias=False)          # W_DKV
        self.kv_norm = build_norm(cfg, dc) if cfg.mla_latent_norm else nn.Identity()
        self.k_up = nn.Linear(dc, H * Dh, bias=False)                  # W_UK (all heads)
        self.v_up = nn.Linear(dc, H * Dh, bias=False)                  # W_UV (all heads)
        self.o_proj = nn.Linear(H * Dh, cfg.d_model, bias=False)
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)
        self.absorb = False                           # runtime switch (inference only)

    @staticmethod
    def cache_len(cache):
        return cache.shape[1]

    def cache_numbers_per_token(self):
        return self.d_c

    def forward(self, x, attn_bias=None, rope_cs=None, kv_cache=None):
        assert rope_cs is None, "MLA is used with ALiBi (or no positions), not RoPE"
        B, T, _ = x.shape
        H, Dh = self.n_head, self.head_dim
        c = self.kv_norm(self.kv_down(x))                         # [B, T, d_c]
        if kv_cache is not None:
            c = torch.cat([kv_cache, c], dim=1)                   # [B, S, d_c]
        new_cache = c
        S = c.shape[1]
        q = self.q_proj(x).view(B, T, H, Dh).transpose(1, 2)      # [B, H, T, Dh]
        is_causal = (attn_bias is None and T > 1)
        dp = self.cfg.attn_dropout if self.training else 0.0

        if not self.absorb:
            k = self.k_up(c).view(B, S, H, Dh).transpose(1, 2)    # [B, H, S, Dh]
            v = self.v_up(c).view(B, S, H, Dh).transpose(1, 2)
            out = attend(q, k, v, attn_bias, is_causal, self.scale, dp, self.cfg.use_sdpa)
        else:
            W_uk = self.k_up.weight.view(H, Dh, self.d_c)         # rows of head h -> W_UK^(h)
            W_uv = self.v_up.weight.view(H, Dh, self.d_c)
            q_lat = torch.einsum("bhtd,hdc->bhtc", q, W_uk.to(q.dtype))   # W_UK^T q
            c_h = c.unsqueeze(1).expand(B, H, S, self.d_c)
            out_lat = attend(q_lat, c_h, c_h, attn_bias, is_causal, self.scale, dp,
                             self.cfg.use_sdpa)                   # [B, H, T, d_c]
            out = torch.einsum("bhtc,hdc->bhtd", out_lat, W_uv.to(out_lat.dtype))

        out = out.transpose(1, 2).contiguous().view(B, T, H * Dh)
        return self.resid_dropout(self.o_proj(out)), new_cache

In [ ]:
# Cell 9 — Channel mixers, registries, transformer block

class SwiGLU(nn.Module):
    """down( silu(gate(x)) * up(x) )."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.up_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)

    def forward(self, x):
        return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class GELUMLP(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.fc1 = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)

    def forward(self, x):
        return self.drop(self.down_proj(F.gelu(self.fc1(x))))


TOKEN_MIXERS: Dict[str, Any] = {}
CHANNEL_MIXERS: Dict[str, Any] = {}

def register_token_mixer(name):
    def deco(fn):
        TOKEN_MIXERS[name] = fn
        return fn
    return deco

def register_channel_mixer(name):
    def deco(fn):
        CHANNEL_MIXERS[name] = fn
        return fn
    return deco

# FIX: token mixers no longer receive `rope` at build time (model 01 silently dropped it there).
# Positional information now flows through forward(): rope_cs / attn_bias, built in the Backbone.
@register_token_mixer("attention")
def _build_attention(cfg):
    return Attention(cfg)

@register_token_mixer("mla")
def _build_mla(cfg):
    return MLAttention(cfg)

@register_channel_mixer("mlp")
def _build_mlp(cfg):
    return {"swiglu": SwiGLU, "gelu": GELUMLP}[cfg.mlp_type](cfg)


class Block(nn.Module):
    """Pre-norm block: x = x + mixer(norm(x)); x = x + mlp(norm(x))."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.norm1 = build_norm(cfg)
        self.mixer = TOKEN_MIXERS[cfg.mixer_type](cfg)
        self.norm2 = build_norm(cfg)
        self.cmix = CHANNEL_MIXERS[cfg.channel_mixer_type](cfg)

    def forward(self, x, **kw):
        h, new_cache = self.mixer(self.norm1(x), **kw)
        x = x + h
        x = x + self.cmix(self.norm2(x))
        return x, new_cache

In [ ]:
# Cell 10 — Backbone and language model

class Backbone(nn.Module):
    """Embedding -> N blocks -> final norm. Builds positional info and masks ONCE per forward."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.emb_drop = nn.Dropout(cfg.embd_dropout)
        self.alibi = ALiBi(cfg.n_head) if cfg.pos_type == "alibi" else None
        self.rope = RoPE(cfg.head_dim, 2 * cfg.max_seq_len, cfg.rope_theta) \
            if cfg.pos_type == "rope" else None
        self.pos_emb = nn.Embedding(2 * cfg.max_seq_len, cfg.d_model) \
            if cfg.pos_type == "learned" else None
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.norm_f = build_norm(cfg)

    def _attn_bias(self, B, T, past, attention_mask, device):
        """Additive mask [B|1, H|1, T, S] or None (=> SDPA fast causal path)."""
        S = past + T
        need = (self.alibi is not None) or (attention_mask is not None) or (past > 0 and T > 1)
        if not need:
            return None
        if self.alibi is not None:
            bias = self.alibi.bias(T, S, device)                          # ALiBi + causal
        else:
            q_pos = torch.arange(past, S, device=device)
            k_pos = torch.arange(S, device=device)
            bias = torch.zeros(T, S, device=device).masked_fill(
                k_pos[None, :] > q_pos[:, None], float("-inf"))[None, None]
        if attention_mask is not None:
            # Mask padded KEYS. A padded QUERY would then have every key masked -> softmax of
            # all -inf -> NaN, which can leak into later layers through 0*NaN. So let each
            # padded query attend to itself (its output is never used).
            key_pad = ~attention_mask.bool()                                   # [B, S]
            q_idx = torch.arange(past, S, device=device)
            k_idx = torch.arange(S, device=device)
            is_self = q_idx[:, None] == k_idx[None, :]                         # [T, S]
            block = key_pad[:, None, :] & ~is_self[None]                       # [B, T, S]
            pad = torch.zeros(B, 1, T, S, device=device).masked_fill(block[:, None], float("-inf"))
            bias = bias + pad
        return bias

    def forward(self, input_ids, attention_mask=None, kv_caches=None):
        B, T = input_ids.shape
        dev = input_ids.device
        past = 0 if kv_caches is None else self.blocks[0].mixer.cache_len(kv_caches[0])
        x = self.tok_emb(input_ids)

        rope_cs = None
        if self.rope is not None or self.pos_emb is not None:
            if attention_mask is not None:       # left padding: real tokens start at position 0
                pos = (attention_mask.long().cumsum(-1) - 1).clamp(min=0)[:, -T:]
            else:
                pos = torch.arange(past, past + T, device=dev).unsqueeze(0).expand(B, T)
            if self.rope is not None:
                rope_cs = self.rope.tables(pos)
            else:
                x = x + self.pos_emb(pos)

        attn_bias = self._attn_bias(B, T, past, attention_mask, dev)
        x = self.emb_drop(x)
        new_caches = []
        for i, blk in enumerate(self.blocks):
            x, nc = blk(x, attn_bias=attn_bias, rope_cs=rope_cs,
                        kv_cache=None if kv_caches is None else kv_caches[i])
            new_caches.append(nc)
        return self.norm_f(x), new_caches


class LegalLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.backbone = Backbone(cfg)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.backbone.tok_emb.weight

    def set_mla_absorb(self, flag: bool):
        for blk in self.backbone.blocks:
            if isinstance(blk.mixer, MLAttention):
                blk.mixer.absorb = flag

    def forward(self, input_ids, targets=None, attention_mask=None, kv_caches=None,
                last_only=False, reduction="mean"):
        h, new_caches = self.backbone(input_ids, attention_mask=attention_mask,
                                      kv_caches=kv_caches)
        if last_only:                 # generation: only the last position's logits are needed
            h = h[:, -1:, :]
        logits = self.lm_head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1),
                                   ignore_index=-100, reduction=reduction)
            if reduction == "none":
                loss = loss.view(targets.shape)
        return logits, loss, new_caches

In [ ]:
# Cell 11 — Init, parameter counting, optimiser groups, checkpoint I/O

def init_weights(model: LegalLM, cfg: ModelConfig):
    """N(0, 0.02) everywhere; residual-branch output projections scaled by 1/sqrt(2L)
    so the residual stream's variance does not grow with depth."""
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
    scale = 1.0 / math.sqrt(2 * cfg.n_layer)
    with torch.no_grad():
        for blk in model.backbone.blocks:
            blk.mixer.o_proj.weight.mul_(scale)
            blk.cmix.down_proj.weight.mul_(scale)


def count_params(model: LegalLM) -> Dict[str, int]:
    seen, total = set(), 0
    for p in model.parameters():
        if id(p) not in seen:
            seen.add(id(p))
            total += p.numel()
    emb = model.backbone.tok_emb.weight.numel()
    attn = sum(p.numel() for blk in model.backbone.blocks for p in blk.mixer.parameters())
    return {"total": total, "embedding": emb, "non_embedding": total - emb, "attention": attn}


def build_param_groups(model: nn.Module, weight_decay: float):
    """Decay only 2-D weight matrices (not embeddings, norm gains or biases)."""
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim >= 2 and "tok_emb" not in name and "pos_emb" not in name:
            decay.append(p)
        else:
            no_decay.append(p)
    return [{"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0}]


def save_model(path, model: LegalLM, cfg: ModelConfig, extra: dict = None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"model": model.state_dict(), "config": asdict(cfg), "extra": extra or {}}, path)


def load_model(path, map_location="cpu") -> Tuple[LegalLM, ModelConfig, dict]:
    ckpt = torch.load(path, map_location=map_location, weights_only=False)
    cfg = ModelConfig(**ckpt["config"])
    model = LegalLM(cfg)
    model.load_state_dict(ckpt["model"])
    return model, cfg, ckpt.get("extra", {})

## Load the Phase-1 checkpoint

In [ ]:
# Cell 12 — Load the Phase-1 checkpoint and sanity-check it
_ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
lm_cfg = ModelConfig(**_ck["config"])
lm = LegalLM(lm_cfg)
lm.load_state_dict(_ck["model"])
lm_extra = _ck.get("extra", {})
del _ck
lm = lm.to(device).eval()
pc = count_params(lm)
print(f"Phase-1 model | pos_type={lm_cfg.pos_type} mixer={lm_cfg.mixer_type} (latent {lm_cfg.kv_latent_dim}) | "
      f"layers={lm_cfg.n_layer} d_model={lm_cfg.d_model} heads={lm_cfg.n_head} d_ff={lm_cfg.d_ff} | "
      f"params {pc['total']/1e6:.2f}M (embedding {pc['embedding']/1e6:.2f}M)")
print("checkpoint extra: step", lm_extra.get("step"), "| val_loss", lm_extra.get("val_loss"))

# If the weights did not load properly, the LM loss on a normal legal sentence would sit near ln(V) ~ 10.4.
_sent = "The appeal is allowed and the impugned judgment of the High Court is set aside."
_ids = [BOS_ID] + tokenizer.encode(_sent)
_x = torch.tensor([_ids[:-1]], device=device); _y = torch.tensor([_ids[1:]], device=device)
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
    _, _l, _ = lm(_x, targets=_y)
print(f"LM loss on a legal sentence: {_l.item():.3f}  (ln V = {math.log(VOCAB_SIZE):.2f})")
assert _l.item() < 9.0, "checkpoint looks untrained / mis-loaded"

if ATTN_MODE in ("bidir", "bidir_top") and lm_cfg.pos_type != "alibi":
    raise RuntimeError(f"ATTN_MODE='{ATTN_MODE}' is implemented for ALiBi checkpoints (pos_type='alibi'); this one has "
                       f"pos_type='{lm_cfg.pos_type}'. Use ATTN_MODE='echo' or 'causal'.")
if METHOD == "partial":
    assert 0 <= PARTIAL_K <= lm_cfg.n_layer, "PARTIAL_K must be between 0 and the number of layers"

## Phase-3 data: labels, documents, tokenisation
Documents are `{"id", "text": [sentences], "labels": [labels]}`. The loader also accepts a few equivalent layouts (dict keyed by id, `sentences` instead of `text`) and **fails loudly** with the offending example if a label is not in `label_vocab.json` or if the numbers of sentences and labels differ. If the schema printed below is not what you expect, paste it to Claude.

In [ ]:
# Cell 13 — Phase-3 data: label vocabulary, documents (CL / IT, train / dev / test)
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

P3 = "Phase_3_RR"
SPEC_LABELS = ["Fact", "Issue", "ArgumentPetitioner", "ArgumentRespondent", "Statute", "Dissent", "PrecedentReliedUpon",
               "PrecedentNotReliedUpon", "PrecedentOverruled", "RulingByLowerCourt", "RatioOfTheDecision", "RulingByPresentCourt", "None"]

def fetch(rel: str) -> Path:
    if LOCAL_DATA_DIR is not None and (Path(LOCAL_DATA_DIR) / rel).exists():
        return Path(LOCAL_DATA_DIR) / rel
    try:
        return Path(hf_hub_download(repo_id=HF_REPO, repo_type="dataset", filename=rel, token=os.environ.get("HF_TOKEN")))
    except Exception as ex:
        raise RuntimeError(f"could not download '{rel}' from {HF_REPO} ({type(ex).__name__}: {ex}). "
                           f"Check the HF login (Cell 0) / internet setting, or download the Phase_3_RR folder and set LOCAL_DATA_DIR.") from ex

def load_json(rel):
    return json.loads(fetch(rel).read_text(encoding="utf-8"))

def parse_label_vocab(raw):
    """label_vocab.json -> ordered list of label names (accepts a list, {label: id}, {id: label} or {"labels": [...]})."""
    if isinstance(raw, dict) and "labels" in raw and isinstance(raw["labels"], (list, dict)):
        raw = raw["labels"]
    if isinstance(raw, list):
        if all(isinstance(x, str) for x in raw): return list(raw)
        if all(isinstance(x, dict) for x in raw):
            out = []
            for x in raw:
                k = next((x[k] for k in ("label", "name", "role") if isinstance(x.get(k), str)), None)
                if k is None: raise ValueError(f"cannot parse label_vocab entry {x!r}")
                out.append(k)
            return out
    if isinstance(raw, dict):
        if all(isinstance(v, int) and not isinstance(v, bool) for v in raw.values()):
            return [k for k, _ in sorted(raw.items(), key=lambda kv: kv[1])]
        if all(isinstance(k, str) and k.isdigit() for k in raw) and all(isinstance(v, str) for v in raw.values()):
            return [raw[k] for k in sorted(raw, key=int)]
        return list(raw)
    raise ValueError(f"cannot parse label_vocab.json: {type(raw).__name__}")

LABELS = parse_label_vocab(load_json(f"{P3}/Train/label_vocab.json"))
if set(LABELS) != set(SPEC_LABELS):
    print("WARNING: label_vocab.json differs from the 13 labels in the capstone document:", LABELS)
NL = len(LABELS); L2I = {l: i for i, l in enumerate(LABELS)}
print(NL, "labels:", LABELS)

def parse_docs(raw, domain, has_labels):
    """-> list of {uid, id, domain, sents, y|None}. uid = 'DOMAIN::id' (unique across domains, used internally only)."""
    if isinstance(raw, dict):
        items = []
        for k, v in raw.items():
            if isinstance(v, list): v = {"text": v}
            if not isinstance(v, dict): raise ValueError(f"unexpected document layout for key {k!r}: {type(v).__name__}")
            v = dict(v); v.setdefault("id", k); items.append(v)
    else:
        items = list(raw)
    docs = []
    for i, e in enumerate(items):
        did = str(next((e[k] for k in ("id", "doc_id", "document_id") if k in e), i))
        sents = e.get("text", e.get("sentences"))
        if not isinstance(sents, list):
            raise ValueError(f"{domain} doc {did}: expected 'text' to be a list of sentences, got {type(sents).__name__}. Keys: {list(e)}")
        sents = [s if isinstance(s, str) else str(s.get("text", "")) for s in sents]
        y = None
        if has_labels:
            labs = e.get("labels", e.get("label", e.get("roles")))
            if not isinstance(labs, list) or len(labs) != len(sents):
                raise ValueError(f"{domain} doc {did}: {len(sents)} sentences but labels = "
                                 f"{None if not isinstance(labs, list) else len(labs)}. Keys: {list(e)}")
            bad = [l for l in labs if l not in L2I]
            if bad: raise ValueError(f"{domain} doc {did}: label(s) {sorted(set(bad))[:5]} not in label_vocab.json {LABELS}")
            y = np.asarray([L2I[l] for l in labs], dtype=np.int64)
        docs.append(dict(uid=f"{domain}::{did}", id=did, domain=domain, sents=sents, y=y))
    return docs

def load_split(domain, split):
    rel = f"{P3}/{'Test' if split == 'test' else 'Train'}/{domain}_{split}.json"
    raw = load_json(rel)
    first = raw[0] if isinstance(raw, list) and raw else (next(iter(raw.values())) if isinstance(raw, dict) and raw else None)
    print(f"  {rel}: {type(raw).__name__} of {len(raw)} | first item keys: {list(first) if isinstance(first, dict) else type(first).__name__}")
    return parse_docs(raw, domain, has_labels=(split != "test"))

TRAIN_ALL = {dom: load_split(dom, "train") for dom in DOMAINS}
DEV_ALL   = {dom: load_split(dom, "dev")   for dom in DOMAINS}
TEST_ALL  = {dom: load_split(dom, "test")  for dom in DOMAINS}

for dom in DOMAINS:
    for name, split in (("train", TRAIN_ALL), ("dev", DEV_ALL), ("test", TEST_ALL)):
        ds = split[dom]; print(f"{dom} {name:5s}: {len(ds):4d} docs | {sum(len(d['sents']) for d in ds):6d} sentences")
    cnt = Counter(LABELS[i] for d in TRAIN_ALL[dom] for i in d["y"])
    print(f"  {dom} train label counts:", dict(cnt.most_common()))
_test_ids = [d["id"] for dom in DOMAINS for d in TEST_ALL[dom]]
if len(set(_test_ids)) != len(_test_ids):
    print("WARNING: test document ids repeat across the domains; the single prediction.json cannot hold both -> paste this to Claude.")

In [ ]:
# Cell 14 — Tokenise the sentences (shared byte-level BPE), domain tags, delimiter, cache on disk
DOMAIN_TEXT = {"CL": "Corporate Law\n", "IT": "Information Technology\n"}
NL_IDS  = list(tokenizer.encode("\n"))                       # sentence delimiter (the Phase-1 corpus kept newlines)
DOM_IDS = {dom: (list(tokenizer.encode(DOMAIN_TEXT.get(dom, dom + "\n"))) if DOMAIN_TAG else []) for dom in DOMAINS}
NLN = len(NL_IDS)
BASE_LEN = {dom: 1 + len(DOM_IDS[dom]) for dom in DOMAINS}   # [BOS] + domain tag
assert NLN >= 1
print("delimiter ids:", NL_IDS, "| domain tag ids:", {k: v for k, v in DOM_IDS.items()})

def tokenize_docs(docs, cache_name):
    """Adds d['tok'] (list of int32 arrays, each truncated to MAX_SENT_TOK, never empty) and d['full_len']. Cached on disk."""
    cdir = WORK_DIR / "cache"; cdir.mkdir(exist_ok=True)
    cp = cdir / f"{cache_name}_v{VOCAB_SIZE}_m{MAX_SENT_TOK}.pkl"
    cache = pickle.load(open(cp, "rb")) if cp.exists() else {}
    changed = False
    for d in tqdm(docs, desc=f"tokenise {cache_name}", leave=False):
        hit = cache.get(d["uid"])
        if hit is None or len(hit[0]) != len(d["sents"]):
            toks, full = [], []
            for s in d["sents"]:
                ids = list(tokenizer.encode(s)); full.append(len(ids))
                ids = ids[:MAX_SENT_TOK] if ids else NL_IDS[:1]          # an empty sentence still needs one token to pool over
                toks.append(np.asarray(ids, dtype=np.int32))
            hit = (toks, full); cache[d["uid"]] = hit; changed = True
        d["tok"], d["full_len"] = hit
    if changed:
        pickle.dump(cache, open(cp, "wb"))
    return docs

def token_stats(docs, title):
    lens = np.concatenate([np.asarray(d["full_len"]) for d in docs]) if docs else np.zeros(1)
    print(f"{title}: {len(lens)} sentences | tokens/sentence mean {lens.mean():.1f}, median {np.median(lens):.0f}, p95 {np.percentile(lens, 95):.0f}, "
          f"max {lens.max()} | truncated at {MAX_SENT_TOK}: {100 * (lens > MAX_SENT_TOK).mean():.2f}%")

## Windows, batches, metrics, bias tuning (pure python / numpy)
`make_windows` packs consecutive sentences into sequences of at most `SEQ_BUDGET` tokens; the next window starts after the first `1 − WINDOW_OVERLAP` of the previous one, so every sentence is covered. With `ATTN_MODE="echo"`, `build_batch` writes the window twice and pools from the second copy (as in the Dec-01 notebook).

In [ ]:
# Cell 15 — Windows and batch layout (with the echo option)
def make_windows(d, budget, overlap, first_budget=None):
    """Sentence index ranges (a, b) so that  BASE + sum(len) + (b-a-1)*NLN <= budget  (at least one sentence per window).
    The next window starts after the first (1-overlap) of this window's tokens; the last window always ends at the last sentence."""
    lens = [len(t) for t in d["tok"]]; base = BASE_LEN[d["domain"]]; n = len(lens)
    out, a, first = [], 0, True
    while a < n:
        bud = first_budget if (first and first_budget is not None) else budget
        tot, b = base, a
        while b < n:
            add = lens[b] + (NLN if b > a else 0)
            if b > a and tot + add > bud: break
            tot += add; b += 1
        out.append((a, b))
        if b >= n: break
        target, adv, a2 = (1.0 - overlap) * (tot - base), 0, a
        while a2 < b - 1 and adv < target:
            adv += lens[a2] + NLN; a2 += 1
        a = max(a2, a + 1); first = False
    return out

def doc_windows(d, rng=None):
    """Training (rng given): randomised first window. Inference (rng None): deterministic."""
    n = len(d["tok"])
    if CONTEXT_MODE == "sentence":
        return [(i, i + 1) for i in range(n)]
    fb = int(SEQ_BUDGET * rng.uniform(0.4, 1.0)) if rng is not None else None
    return make_windows(d, SEQ_BUDGET, WINDOW_OVERLAP, fb)

def build_batch(items, docs, with_labels=True):
    """items: [(doc_index, a, b)].  Returns ids, mask, seg, y, owners, n_sent.
       seg[b, t] = global sentence number (0..n_sent-1) of the token, or -1 for BOS / domain tag / delimiters / EOS / first copy / padding.
       causal / bidir :  [BOS] tag s_a ⏎ s_a+1 ...
       echo           :  [BOS] tag s_a ⏎ ... [EOS] [BOS] tag s_a ⏎ ...   (sentences are read from the SECOND copy, which has seen the whole window)
       owners[g] = (doc_index, sentence_index, a, b) for global sentence g."""
    seqs, segs, ys, owners, g = [], [], [], [], 0
    for di, a, b in items:
        d = docs[di]; head = [BOS_ID] + DOM_IDS[d["domain"]]
        body, bseg = [], []
        for j in range(a, b):
            if j > a:
                body += NL_IDS; bseg += [-1] * NLN
            t = d["tok"][j].tolist()
            body += t; bseg += [g] * len(t)
            owners.append((di, j, a, b))
            if with_labels: ys.append(int(d["y"][j]))
            g += 1
        if ATTN_MODE == "echo":
            seq = head + body + [EOS_ID] + head + body
            seg = [-1] * (len(head) + len(body) + 1 + len(head)) + bseg
        else:
            seq = head + body
            seg = [-1] * len(head) + bseg
        assert len(seq) <= 1024, f"sequence of {len(seq)} tokens exceeds the Phase-1 context"
        seqs.append(seq); segs.append(seg)
    L = max(len(s) for s in seqs)
    ids  = np.full((len(seqs), L), PAD_ID, dtype=np.int64)
    mask = np.zeros((len(seqs), L), dtype=np.int64)
    sg   = np.full((len(seqs), L), -1, dtype=np.int64)
    for i, (s, q) in enumerate(zip(seqs, segs)):
        ids[i, :len(s)] = s; mask[i, :len(s)] = 1; sg[i, :len(q)] = q
    y = np.asarray(ys, dtype=np.int64) if with_labels else None
    return ids, mask, sg, y, owners, g

In [ ]:
# Cell 16 — Metrics (macro-F1, per domain), confusion, per-class bias tuning
def confusion(y, p):
    cm = np.zeros((NL, NL), dtype=np.int64); np.add.at(cm, (y, p), 1); return cm

def scores_from_cm(cm):
    """cm[gold, pred].  macro_f1 averages over labels that occur in gold OR prediction (sklearn's default);
    macro_f1_gold averages over labels present in the gold only."""
    tp = np.diag(cm).astype(np.float64); sup = cm.sum(1).astype(np.float64); npred = cm.sum(0).astype(np.float64)
    P = np.divide(tp, npred, out=np.zeros(NL), where=npred > 0)
    R = np.divide(tp, sup,   out=np.zeros(NL), where=sup > 0)
    F = np.divide(2 * P * R, P + R, out=np.zeros(NL), where=(P + R) > 0)
    union, gold = (sup + npred) > 0, sup > 0
    return dict(P=P, R=R, F=F, support=sup, cm=cm, accuracy=float(tp.sum() / max(cm.sum(), 1)),
                macro_f1=float(F[union].mean()), macro_p=float(P[union].mean()), macro_r=float(R[union].mean()),
                macro_f1_gold=float(F[gold].mean()))

def evaluate(docs, probs, bias=None):
    """docs with gold labels + per-doc [n_sent, NL] probabilities -> {domain: scores, 'ALL': scores, 'select': mean of per-domain macro-F1}."""
    ys, ps = defaultdict(list), defaultdict(list)
    for d, p in zip(docs, probs):
        lp = np.log(np.maximum(p, 1e-8)) + (0.0 if bias is None else np.asarray(bias))
        ys[d["domain"]].append(d["y"]); ps[d["domain"]].append(lp.argmax(1))
    out = {dom: scores_from_cm(confusion(np.concatenate(ys[dom]), np.concatenate(ps[dom]))) for dom in ys}
    out["ALL"] = scores_from_cm(confusion(np.concatenate([np.concatenate(v) for v in ys.values()]),
                                          np.concatenate([np.concatenate(v) for v in ps.values()])))
    out["select"] = float(np.mean([out[dom]["macro_f1"] for dom in ys]))
    return out

def tune_bias(docs, probs, grid=np.arange(-1.5, 1.51, 0.25), passes=2):
    """Coordinate ascent on a per-class additive log-probability bias, objective = mean per-domain macro-F1 on these docs."""
    LP = np.concatenate([np.log(np.maximum(p, 1e-8)) for p in probs])
    Y  = np.concatenate([d["y"] for d in docs])
    DM = np.concatenate([[d["domain"]] * len(d["y"]) for d in docs])
    doms = sorted(set(DM.tolist()))
    def obj(b):
        pred = (LP + b).argmax(1)
        return float(np.mean([scores_from_cm(confusion(Y[DM == dm], pred[DM == dm]))["macro_f1"] for dm in doms]))
    b = np.zeros(NL); base = best = obj(b)
    for _ in range(passes):
        for c in range(NL):
            keep = b[c]
            for v in grid:
                b2 = b.copy(); b2[c] = v; s = obj(b2)
                if s > best + 1e-9: best, keep = s, v
            b[c] = keep
    return b, base, best

def cv_bias_gain(docs, probs, folds=2):
    """Honest check of the bias tuning: tune on half of the dev DOCUMENTS, score the other half (and vice versa); mean held-out gain."""
    gains = []
    for f in range(folds):
        tr = [i for i in range(len(docs)) if i % folds != f]; te = [i for i in range(len(docs)) if i % folds == f]
        b, _, _ = tune_bias([docs[i] for i in tr], [probs[i] for i in tr])
        base  = evaluate([docs[i] for i in te], [probs[i] for i in te])["select"]
        tuned = evaluate([docs[i] for i in te], [probs[i] for i in te], b)["select"]
        gains.append(tuned - base)
    return float(np.mean(gains))

def top_confusions(cm, k=10):
    pairs = [(int(cm[i, j]), LABELS[i], LABELS[j]) for i in range(NL) for j in range(NL) if i != j and cm[i, j] > 0]
    return sorted(pairs, reverse=True)[:k]

## Model, adaptation methods (rsLoRA / LoRA / partial / full / norm-only), training
The attention modes and `partial` are the Phase-2 Dec-03 code; the head pools sentence tokens instead of tagging tokens. **New:** the `rank_stabilized` switch of the LoRA layer (the whole of rsLoRA) and the scaling demo.

In [ ]:
# Cell 17 — Attention modes, the RR model, LoRA / rsLoRA, and the adaptation methods
import copy

# ---- (a) Attention modes ---------------------------------------------------------------------------------------------------
# Phase-1 Backbone builds ONE additive bias [B,H,T,S] = ALiBi (-m_h (i-j)) + causal (-inf for j>i) + padding, and gives it to
# every block. Here we replace Backbone.forward by a version that can hand DIFFERENT biases to different blocks:
#   blocks < bidir_from : the original causal ALiBi bias           (frozen lower blocks keep their Phase-1 behaviour)
#   blocks >= bidir_from: symmetric ALiBi  -m_h |i-j|, no causal part (right context is visible)
def _bidir_alibi_bias(bb, B, T, attention_mask, dev):
    pos = torch.arange(T, device=dev)
    dist = (pos[:, None] - pos[None, :]).abs().float()                           # |i - j|
    bias = (-bb.alibi.slopes.to(dev)[:, None, None] * dist).unsqueeze(0)         # [1, H, T, T]
    if attention_mask is not None:                                               # mask padded KEYS (real keys always remain, so no NaN rows)
        bias = bias.expand(B, -1, -1, -1).masked_fill(~attention_mask.bool()[:, None, None, :], float("-inf"))
    return bias

def _hybrid_forward(self, input_ids, attention_mask=None, kv_caches=None):
    assert kv_caches is None, "the NER notebook never uses a KV cache"
    B, T = input_ids.shape
    dev = input_ids.device
    x = self.emb_drop(self.tok_emb(input_ids))
    bias_causal = Backbone._attn_bias(self, B, T, 0, attention_mask, dev) if self.bidir_from > 0 else None
    bias_bidir  = _bidir_alibi_bias(self, B, T, attention_mask, dev) if self.bidir_from < len(self.blocks) else None
    for i, blk in enumerate(self.blocks):
        x, _ = blk(x, attn_bias=bias_bidir if i >= self.bidir_from else bias_causal, rope_cs=None, kv_cache=None)
    return self.norm_f(x), []

def set_attention_mode(bb):
    """Install the attention mode on a Backbone instance. causal / echo keep the untouched Phase-1 forward."""
    n = len(bb.blocks)
    if ATTN_MODE == "bidir":
        bb.bidir_from = 0
    elif ATTN_MODE == "bidir_top":
        bb.bidir_from = max(0, n - BIDIR_TOP_K)
    else:
        return
    bb.forward = types.MethodType(_hybrid_forward, bb)
    print(f"attention: blocks 0..{bb.bidir_from - 1} causal | blocks {bb.bidir_from}..{n - 1} bidirectional (symmetric ALiBi)")



# ---- (b) RR model: backbone + mean-pool over each sentence's tokens + Linear head ----------------------------------------------
class RRModel(nn.Module):
    """Backbone (embedding -> 12 blocks -> final RMSNorm), mean over the tokens of each sentence, dropout, Linear(d_model -> n_labels).
    The sentence-level version of the Module 2.4 recipe (one extra output layer on the final hidden vectors)."""
    def __init__(self, backbone, d_model, n_labels, dropout):
        super().__init__()
        self.backbone = backbone
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(d_model, n_labels)
        nn.init.normal_(self.head.weight, 0.0, 0.02); nn.init.zeros_(self.head.bias)

    def forward(self, input_ids, attention_mask, seg, n_sent):
        h, _ = self.backbone(input_ids, attention_mask=attention_mask)          # [B, T, d]
        h = h.float()
        valid = seg >= 0
        idx = seg[valid]                                                         # global sentence number of every sentence token
        pooled = h.new_zeros(n_sent, h.size(-1)).index_add_(0, idx, h[valid])   # sum of the token vectors of each sentence
        cnt = torch.bincount(idx, minlength=n_sent).clamp_min(1).unsqueeze(1).to(pooled.dtype)
        return self.head(self.drop(pooled / cnt))                                # [n_sent, n_labels]



# ---- (c) LoRA and rank-stabilised LoRA ---------------------------------------------------------------------------------------
class LoRALinear(nn.Module):
    """y = W x + gamma * B(A(dropout(x))).  W frozen; A ~ Kaiming-uniform, B = 0, so the update starts at exactly 0.
       standard LoRA:  gamma = alpha / r
       rsLoRA:         gamma = alpha / sqrt(r)   (Kalajdzievski 2023: the unique scaling for which learning is stable for every rank)"""
    def __init__(self, base: nn.Linear, r: int, alpha: float, dropout: float, rank_stabilized: bool = False):
        super().__init__()
        self.base, self.r = base, r
        self.scale = alpha / math.sqrt(r) if rank_stabilized else alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.t() @ self.B.t()) * self.scale


def apply_adapter(module, targets, make):
    """Replace every nn.Linear whose attribute name is in `targets` by make(linear). Returns how many were replaced."""
    n = 0
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and name in targets:
            setattr(module, name, make(child)); n += 1
        else:
            n += apply_adapter(child, targets, make)
    return n



def scaling_demo(ranks=(4, 16, 64, 256), d=512, alpha=16.0, eta=1.0, trials=400, seed=0):
    """One SGD step on B from B = 0 with a linear loss L = g.h:  Delta h = -eta * gamma^2 * ||A x||^2 * g, with ||A x||^2 ~ r.
    Prints |Delta h| (relative to rank 4) for gamma = alpha/r (LoRA) and gamma = alpha/sqrt(r) (rsLoRA)."""
    gen = torch.Generator().manual_seed(seed)
    bound = 1.0 / math.sqrt(d)                                   # Kaiming-uniform(a=sqrt(5)) bound for fan_in = d
    rows = []
    for r in ranks:
        lo, rs = [], []
        for _ in range(trials):
            x = torch.randn(d, generator=gen); g = torch.randn(d, generator=gen); g = g / g.norm()
            A = (torch.rand(r, d, generator=gen) * 2 - 1) * bound
            Ax2 = float((A @ x).pow(2).sum())
            lo.append(eta * (alpha / r) ** 2 * Ax2); rs.append(eta * (alpha / math.sqrt(r)) ** 2 * Ax2)   # |Delta h| (|g| = 1)
        rows.append((r, float(np.mean(lo)), float(np.mean(rs))))
    b_lo, b_rs = rows[0][1], rows[0][2]
    print("one-step output change |dh| relative to rank 4  (theory: LoRA ~ 1/r, rsLoRA ~ constant)")
    print(f"  {'rank r':>7s} | {'LoRA  gamma=a/r':>16s} {'|dh| rel.':>10s} | {'rsLoRA gamma=a/sqrt(r)':>23s} {'|dh| rel.':>10s}")
    for r, lo, rs in rows:
        print(f"  {r:7d} | {alpha / r:16.4f} {lo / b_lo:10.3f} | {alpha / math.sqrt(r):23.4f} {rs / b_rs:10.3f}")

if SCALING_DEMO:
    scaling_demo()


def build_rr_model():
    """Fresh copy of the Phase-1 weights every call (several runs in one session do not contaminate each other)."""
    backbone = copy.deepcopy(lm.backbone).cpu()
    ref = copy.deepcopy(backbone) if METHOD in ("lora", "rslora") else None    # untouched copy for the identity check
    set_attention_mode(backbone)
    if ref is not None: set_attention_mode(ref)
    model = RRModel(backbone, lm_cfg.d_model, NL, HEAD_DROPOUT)
    hp = HP[METHOD]
    if METHOD == "full":
        pass                                                                    # everything trainable
    elif METHOD == "partial":
        n = len(backbone.blocks)
        for p in backbone.parameters(): p.requires_grad = False                 # embeddings + all blocks frozen ...
        for blk in backbone.blocks[n - PARTIAL_K:]:                             # ... except the top K blocks
            for p in blk.parameters(): p.requires_grad = True
        for p in backbone.norm_f.parameters(): p.requires_grad = True           # final RMSNorm feeds the head
        print(f"partial fine-tuning: blocks {n - PARTIAL_K}..{n - 1} ({PARTIAL_K} of {n}) + final norm + head trainable; "
              f"embeddings and blocks 0..{n - PARTIAL_K - 1} frozen")
        if ATTN_MODE == "bidir":
            print("WARNING: ATTN_MODE='bidir' unmasks the FROZEN lower blocks too (they see an attention pattern they were never "
                  "trained for and cannot adapt). 'bidir_top' keeps them exactly as in Phase 1.")
    elif METHOD in ("lora", "rslora"):
        for p in backbone.parameters(): p.requires_grad = False
        rs = METHOD == "rslora"
        n = apply_adapter(backbone, ADAPTER_TARGETS, lambda l: LoRALinear(l, hp["r"], hp["alpha"], hp["dropout"], rank_stabilized=rs))
        gamma = hp["alpha"] / math.sqrt(hp["r"]) if rs else hp["alpha"] / hp["r"]
        print(f"{'rsLoRA' if rs else 'LoRA'} applied to {n} Linear layers (r={hp['r']}, alpha={hp['alpha']}, gamma={gamma:.4f})")
        # The adapted model must equal the Phase-1 model at initialisation (B = 0).
        _ids = torch.randint(5, VOCAB_SIZE, (1, 48)); _m = torch.ones_like(_ids)
        with torch.no_grad():
            a, _ = backbone.eval()(_ids, attention_mask=_m); b, _ = ref.eval()(_ids, attention_mask=_m)
        err = (a - b).abs().max().item()
        print(f"identity-at-init check: max|adapted - Phase-1| = {err:.2e}")
        assert err < 1e-4, "adapter does not start from the Phase-1 function"
        backbone.train(); del ref
    elif METHOD == "norm_only":
        for p in backbone.parameters(): p.requires_grad = False
        n = 0
        for m in backbone.modules():
            if isinstance(m, (RMSNorm, LayerNorm)):
                for p in m.parameters(): p.requires_grad = True; n += p.numel()
        print(f"norm-only: {n:,} gain parameters unfrozen")
    model.to(device)
    tot = sum(p.numel() for p in model.parameters()); tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"params: total {tot/1e6:.2f}M | trainable {tr/1e6:.3f}M ({100*tr/tot:.2f}%)")
    model.n_total, model.n_trainable = tot, tr
    return model


def class_weights(train_docs):
    cnt = np.zeros(NL)
    for d in train_docs: cnt += np.bincount(d["y"], minlength=NL)
    w = np.ones(NL)
    if CLASS_WEIGHT == "sqrt_inv":
        nz = cnt > 0
        w[nz] = np.clip(np.sqrt(cnt.max() / cnt[nz]), 1.0, MAX_CLASS_W)
    return torch.tensor(w, dtype=torch.float32, device=device), cnt

In [ ]:
# Cell 18 — Inference: windows -> probabilities -> weighted average per sentence
@torch.no_grad()
def predict_probs(model, docs, batch_size=None):
    """Per-doc [n_sentences, NL] probabilities. A sentence covered by several windows gets the weighted average of its probabilities
    (weight = 1 + distance (in sentences) to the nearest window edge when EDGE_WEIGHT, else 1)."""
    model.eval()
    bs = batch_size or (16 if CONTEXT_MODE == "window" else 128)
    jobs = [(di, a, b) for di, d in enumerate(docs) for (a, b) in doc_windows(d)]
    jobs.sort(key=lambda j: sum(len(t) for t in docs[j[0]]["tok"][j[1]:j[2]]))        # similar lengths together
    acc = [np.zeros((len(d["tok"]), NL), dtype=np.float32) for d in docs]
    wsum = [np.zeros(len(d["tok"]), dtype=np.float32) for d in docs]
    for i in range(0, len(jobs), bs):
        ids, mask, seg, _, owners, n_sent = build_batch(jobs[i:i + bs], docs, with_labels=False)
        ids, mask, seg = (torch.from_numpy(x).to(device) for x in (ids, mask, seg))
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = model(ids, mask, seg, n_sent)
        p = logits.float().softmax(-1).cpu().numpy()
        for g, (di, j, a, b) in enumerate(owners):
            w = 1.0 + min(j - a, b - 1 - j) if EDGE_WEIGHT else 1.0
            acc[di][j] += w * p[g]; wsum[di][j] += w
    return [acc[i] / np.maximum(wsum[i], 1e-8)[:, None] for i in range(len(docs))]

In [ ]:
# Cell 19 — Training loop (AdamW, linear warm-up + cosine, grad accumulation, fp16/bf16 autocast)
def make_epoch_items(train_docs, rng):
    items = []
    for di, d in enumerate(train_docs):
        items += [(di, a, b) for a, b in doc_windows(d, rng)]
    rng.shuffle(items)
    return items

def train_rr(model, train_docs, val_docs, epochs, max_steps=None):
    hp = HP[METHOD]
    cw, counts = class_weights(train_docs)
    print("class counts:", dict(zip(LABELS, counts.astype(int).tolist())))
    print("class weights:", dict(zip(LABELS, np.round(cw.cpu().numpy(), 2).tolist())))
    head_p = [p for n, p in model.named_parameters() if n.startswith("head.")]
    dec, nodec = [], []
    for n, p in model.named_parameters():
        if n.startswith("head.") or not p.requires_grad: continue
        (dec if (p.ndim >= 2 and "tok_emb" not in n) else nodec).append(p)
    groups = [dict(params=head_p, lr=hp["head_lr"], base_lr=hp["head_lr"], weight_decay=0.0)]
    if dec:   groups.append(dict(params=dec,   lr=hp["lr"], base_lr=hp["lr"], weight_decay=hp["wd"]))
    if nodec: groups.append(dict(params=nodec, lr=hp["lr"], base_lr=hp["lr"], weight_decay=0.0))
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
    trainables = [p for p in model.parameters() if p.requires_grad]

    n_items = len(make_epoch_items(train_docs, random.Random(SEED + 1)))
    steps_per_epoch = math.ceil(math.ceil(n_items / BATCH) / ACCUM)
    total = epochs * steps_per_epoch
    if max_steps: total = min(total, max_steps)
    warm = max(1, int(WARMUP_FRAC * total))
    def lr_mult(s):
        if s < warm: return (s + 1) / warm
        prog = (s - warm) / max(1, total - warm)
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, prog)))
    print(f"{n_items} sequences/epoch | {steps_per_epoch} optimiser steps/epoch | total {total} steps")

    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    history, losses, step, done = [], [], 0, False
    best = dict(score=-1.0, epoch=0, state=None)
    t_train = time.time()
    for ep in range(1, epochs + 1):
        items = make_epoch_items(train_docs, random.Random(SEED + ep))
        model.train(); t0 = time.time(); micro, ep_loss, ep_n = 0, 0.0, 0
        nb = math.ceil(len(items) / BATCH)
        for bi in range(nb):
            chunk = items[bi * BATCH:(bi + 1) * BATCH]
            ids, mask, seg, y, _, n_sent = build_batch(chunk, train_docs)
            ids, mask, seg, y = (torch.from_numpy(x).to(device) for x in (ids, mask, seg, y))
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
                logits = model(ids, mask, seg, n_sent)
            loss = F.cross_entropy(logits.float(), y, weight=cw, label_smoothing=LABEL_SMOOTHING)
            scaler.scale(loss / ACCUM).backward()
            micro += 1; lv = loss.item(); losses.append(lv); ep_loss += lv; ep_n += 1
            if micro % ACCUM == 0 or bi == nb - 1:
                m = lr_mult(step)
                for g in opt.param_groups: g["lr"] = g["base_lr"] * m
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(trainables, GRAD_CLIP)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
                if step % 25 == 0:
                    print(f"  ep {ep} step {step}/{total} | loss {np.mean(losses[-25*ACCUM:]):.4f} | lr x{m:.3f} | {time.time()-t0:.0f}s")
                if max_steps and step >= max_steps: done = True; break
        rec = dict(epoch=ep, train_loss=ep_loss / max(ep_n, 1), seconds=time.time() - t0, steps=step)
        if val_docs:
            ev = evaluate(val_docs, predict_probs(model, val_docs))
            rec.update(val_select=ev["select"], val_macro_f1_pooled=ev["ALL"]["macro_f1"], val_acc=ev["ALL"]["accuracy"])
            for dom in DOMAINS:
                if dom in ev: rec[f"val_{dom}_macro_f1"] = ev[dom]["macro_f1"]
            if ev["select"] > best["score"]:
                best = dict(score=ev["select"], epoch=ep,
                            state={n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad})
        print(f"epoch {ep}: " + " | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items() if k != "epoch"))
        history.append(rec)
        if done: break
    stats = dict(train_seconds=time.time() - t_train, steps=step,
                 peak_gpu_gb=(torch.cuda.max_memory_allocated() / 2**30) if device.type == "cuda" else 0.0,
                 first_losses=float(np.mean(losses[:20])) if losses else None, last_losses=float(np.mean(losses[-20:])) if losses else None)
    if best["state"] is None:                                    # no validation: keep the final weights
        best = dict(score=None, epoch=history[-1]["epoch"],
                    state={n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad})
    return history, best, stats

In [ ]:
# Cell 20 — Build data, build the model, train
seed_everything(SEED)
if FINAL_FIT:
    train_docs = [d for dom in DOMAINS for d in (TRAIN_ALL[dom] + DEV_ALL[dom])]; val_docs = []
else:
    train_docs = [d for dom in DOMAINS for d in TRAIN_ALL[dom]]
    val_docs = [d for dom in DOMAINS for d in (DEV_ALL[dom][:SMOKE_VAL_DOCS] if SMOKE_TEST else DEV_ALL[dom])]
for dom in DOMAINS:
    tokenize_docs(TRAIN_ALL[dom], f"train_{dom}")
    tokenize_docs(DEV_ALL[dom], f"dev_{dom}")
token_stats(train_docs, "train")
if val_docs: token_stats(val_docs, "dev")
n_win = sum(len(doc_windows(d)) for d in train_docs)
print(f"train docs {len(train_docs)} | dev docs {len(val_docs)} | {n_win} sequences per epoch ({CONTEXT_MODE} mode)")

# reference point: always predicting the most frequent training label
if val_docs:
    _maj = int(np.bincount(np.concatenate([d["y"] for d in train_docs]), minlength=NL).argmax())
    _ev = evaluate(val_docs, [np.eye(NL, dtype=np.float32)[np.full(len(d["y"]), _maj)] for d in val_docs])
    print(f"majority-class baseline ('{LABELS[_maj]}'): dev macro-F1 per-domain mean {_ev['select']:.4f} | accuracy {_ev['ALL']['accuracy']:.3f}")

model = build_rr_model()
history, best, stats = train_rr(model, train_docs, val_docs, epochs=EPOCHS, max_steps=SMOKE_STEPS if SMOKE_TEST else None)
model.load_state_dict(best["state"], strict=False)           # best epoch (or final weights if no validation)
print(f"\nbest epoch {best['epoch']} | dev select-score {best['score']} | train {stats['train_seconds']/60:.1f} min | peak GPU {stats['peak_gpu_gb']:.1f} GB")

In [ ]:
# Cell 21 — Smoke-test verdict (SMOKE_TEST=True only)
if SMOKE_TEST and val_docs:
    drop = stats["last_losses"] / stats["first_losses"]
    probs = predict_probs(model, val_docs); ev = evaluate(val_docs, probs)
    n_pred_labels = len(set(np.concatenate([p.argmax(1) for p in probs]).tolist()))
    print(f"loss {stats['first_losses']:.3f} -> {stats['last_losses']:.3f} (x{drop:.2f}) | dev select-score {ev['select']:.4f} | "
          f"predicted labels in use: {n_pred_labels}/{NL}")
    ok = drop < 0.9 and n_pred_labels >= 3
    print("SMOKE VERDICT:", "OK, the pipeline learns -> run the full job" if ok else
          "NOT OK: loss did not fall clearly or one label dominates; with ~250 steps an adapter can be slow, so compare with METHOD='full' before debugging")
else:
    print("not a smoke run")

In [ ]:
# Cell 22 — Dev analysis: per-domain scores, per-label table, confusions, bias tuning (full runs with a dev set)
BIAS = np.zeros(NL) if BIAS_FINAL is None else np.asarray(BIAS_FINAL, dtype=np.float64)
dev_ev = dev_ev_tuned = None; bias_gain = 0.0
if not SMOKE_TEST and val_docs:
    probs_val = predict_probs(model, val_docs)
    dev_ev = evaluate(val_docs, probs_val)
    for key in [*DOMAINS, "ALL"]:
        s = dev_ev[key]
        print(f"\n===== {key} | macro-F1 {s['macro_f1']:.4f} (gold-labels-only {s['macro_f1_gold']:.4f}) | macro-P {s['macro_p']:.4f} | "
              f"macro-R {s['macro_r']:.4f} | accuracy {s['accuracy']:.4f}")
        print(f"  {'label':24s} {'support':>8s} {'P':>7s} {'R':>7s} {'F1':>7s}")
        for i, l in enumerate(LABELS):
            print(f"  {l:24s} {int(s['support'][i]):8d} {s['P'][i]:7.3f} {s['R'][i]:7.3f} {s['F'][i]:7.3f}")
        print("  most frequent confusions (gold -> predicted):", [f"{g}->{p} x{n}" for n, g, p in top_confusions(s["cm"], 8)])
    print(f"\nselect-score (mean of per-domain macro-F1), UNTUNED: {dev_ev['select']:.4f}   <- report this one")
    if BIAS_TUNE:
        bias_t, b0, b1 = tune_bias(val_docs, probs_val)
        cv_gain = cv_bias_gain(val_docs, probs_val) if len(val_docs) >= 4 else 0.0
        bias_gain = cv_gain
        print(f"bias tuning on dev: {b0:.4f} -> {b1:.4f} (in-sample gain {b1 - b0:+.4f}, optimistic) | "
              f"2-fold document-level held-out gain {cv_gain:+.4f} | bias = {np.round(bias_t, 2).tolist()}")
        if cv_gain >= BIAS_MIN_GAIN:
            BIAS = bias_t; dev_ev_tuned = evaluate(val_docs, probs_val, BIAS)
            print(f"  kept: held-out gain >= BIAS_MIN_GAIN={BIAS_MIN_GAIN}. For FINAL_FIT=True set  BIAS_FINAL = {np.round(BIAS, 2).tolist()}")
        else:
            print(f"  held-out gain below BIAS_MIN_GAIN={BIAS_MIN_GAIN}: bias NOT used")
else:
    print("no dev analysis (smoke run or FINAL_FIT)" + ("" if BIAS_FINAL is None else f" | using BIAS_FINAL={np.round(BIAS, 2).tolist()}"))

In [ ]:
# Cell 23 — Test predictions -> prediction.json -> submission.zip (full runs only)
def write_submission(docs, probs, bias, out_dir):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    sub = {}
    for d, p in zip(docs, probs):
        pred = [LABELS[i] for i in (np.log(np.maximum(p, 1e-8)) + bias).argmax(1)]
        assert len(pred) == len(d["sents"]), f"{d['id']}: {len(pred)} predictions for {len(d['sents'])} sentences"
        assert d["id"] not in sub, f"document id {d['id']!r} occurs in more than one domain; one prediction.json cannot hold both"
        sub[d["id"]] = pred
    (out_dir / "prediction.json").write_text(json.dumps(sub, ensure_ascii=False))
    with zipfile.ZipFile(out_dir / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(out_dir / "prediction.json", arcname="prediction.json")          # must sit at the ROOT of the zip
    n = sum(len(v) for v in sub.values())
    print(f"wrote {out_dir/'submission.zip'} | {len(sub)} docs | {n} sentences | predicted labels:",
          dict(Counter(l for v in sub.values() for l in v).most_common()))
    verify_submission(out_dir / "submission.zip", docs)
    return sub

def verify_submission(zip_path, docs):
    """Re-open the zip like the evaluator would: root file name, ids, one official label per sentence, same order and length."""
    zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        assert names == ["prediction.json"], f"the zip must contain ONLY prediction.json at its root, found {names}"
        sub = json.loads(z.read("prediction.json"))
    assert isinstance(sub, dict), "prediction.json must be a JSON object {doc_id: [label, ...]}"
    want = {d["id"]: len(d["sents"]) for d in docs}
    missing, unknown = set(want) - set(sub), set(sub) - set(want)
    assert not missing and not unknown, f"ids differ from the Phase-3 test set: {len(missing)} missing, {len(unknown)} unknown"
    for did, lst in sub.items():
        assert isinstance(lst, list) and len(lst) == want[did], f"{did}: need {want[did]} labels, got {len(lst) if isinstance(lst, list) else lst!r}"
        assert all(l in L2I for l in lst), f"{did}: label outside label_vocab.json: {sorted(set(lst) - set(L2I))}"
    print(f"VERIFIED {zip_path.resolve()} | {len(sub)} documents | first ids {list(sub)[:3]}")
    print("   -> upload exactly THIS file to the Phase-3 Codabench page (earlier phases produced files with the same name).")

if not SMOKE_TEST:
    for dom in DOMAINS: tokenize_docs(TEST_ALL[dom], f"test_{dom}")
    test_docs = [d for dom in DOMAINS for d in TEST_ALL[dom]]
    token_stats(test_docs, "test")
    probs_test = predict_probs(model, test_docs)
    submission = write_submission(test_docs, probs_test, BIAS, RUN_DIR / "submission")
    for dom in DOMAINS:
        c = Counter(l for d, p in zip(test_docs, probs_test) if d["domain"] == dom for l in (np.log(np.maximum(p, 1e-8)) + BIAS).argmax(1))
        print(f"  {dom} predicted label share:", {LABELS[k]: round(v / sum(c.values()), 3) for k, v in c.most_common()})
else:
    print("smoke run: no submission written")

In [ ]:
# Cell 24 — Save trainable weights + the run log (for the report's compute / memory / PEFT-vs-full / context tables)
def _jsonable(o):
    if isinstance(o, dict): return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (np.floating, np.integer)): return o.item()
    return o

def _ev_summary(ev):
    if ev is None: return None
    return {k: (dict(macro_f1=v["macro_f1"], macro_f1_gold=v["macro_f1_gold"], macro_p=v["macro_p"], macro_r=v["macro_r"], accuracy=v["accuracy"],
                     per_label={LABELS[i]: dict(support=int(v["support"][i]), P=round(float(v["P"][i]), 4), R=round(float(v["R"][i]), 4),
                                                F1=round(float(v["F"][i]), 4)) for i in range(NL)},
                     confusion=v["cm"].tolist(), top_confusions=top_confusions(v["cm"], 12)) if isinstance(v, dict) else v)
            for k, v in ev.items()}

log = dict(run=RUN_NAME, method=METHOD, attn_mode=ATTN_MODE, context_mode=CONTEXT_MODE, window_tokens=WINDOW_TOKENS, window_overlap=WINDOW_OVERLAP,
           max_sent_tok=MAX_SENT_TOK, edge_weight=EDGE_WEIGHT, domains=DOMAINS, domain_tag=DOMAIN_TAG, final_fit=FINAL_FIT, smoke=SMOKE_TEST,
           mixer=lm_cfg.mixer_type, n_layer=lm_cfg.n_layer, partial_k=PARTIAL_K, bidir_top_k=BIDIR_TOP_K, seq_budget=SEQ_BUDGET, pos_type=lm_cfg.pos_type, hp=HP[METHOD], epochs=EPOCHS, batch=BATCH, accum=ACCUM,
           class_weight=CLASS_WEIGHT, labels=LABELS, params_total=model.n_total, params_trainable=model.n_trainable,
           history=history, best_epoch=best["epoch"], stats=stats, bias=BIAS.tolist(), bias_gain=bias_gain,
           dev_untuned=_ev_summary(dev_ev), dev_tuned=_ev_summary(dev_ev_tuned))
(RUN_DIR / "run_log.json").write_text(json.dumps(_jsonable(log), indent=2))
torch.save({"trainable": best["state"], "method": METHOD, "attn_mode": ATTN_MODE, "labels": LABELS, "log": _jsonable(log)},
           RUN_DIR / f"rr_{RUN_NAME}.pt")
print("saved:", sorted(p.name for p in RUN_DIR.iterdir()))

In [ ]:
# Cell 25 — Export this run's per-sentence distributions ("soft labels") so it can serve as the KD teacher (run #4)
# All four Phase-1 models share one tokenizer and the label list is fixed, so these arrays can teach ANY student without this
# notebook's model code. Stored as float16 log-probabilities: softmax(logp / T) = softmax(z / T) because logp = z - logsumexp(z).
if EXPORT_SOFT_LABELS and not SMOKE_TEST:
    _t0 = time.time()
    _exp = {"train": [d for d in train_docs], "dev": [d for d in val_docs]}
    _out, _split = {}, {}
    for _name, _docs in _exp.items():
        if not _docs: continue
        for d, p in zip(_docs, predict_probs(model, _docs)):
            _out[d["uid"]] = np.log(np.maximum(p, 1e-8)).astype(np.float16); _split[d["uid"]] = _name
    _meta = dict(teacher=RUN_NAME, method=METHOD, attn_mode=ATTN_MODE, context_mode=CONTEXT_MODE, window_tokens=WINDOW_TOKENS,
                 params_total=int(model.n_total), params_trainable=int(model.n_trainable), epochs=int(EPOCHS), best_epoch=int(best["epoch"]),
                 final_fit=FINAL_FIT, dev_select=None if dev_ev is None else float(dev_ev["select"]))
    _path = RUN_DIR / f"teacher_softlabels_{RUN_NAME}.pt"
    torch.save(dict(meta=_meta, labels=list(LABELS), docs=_out, split=_split), _path)
    print(f"saved {_path} ({_path.stat().st_size/1e6:.0f} MB) in {time.time()-_t0:.0f}s | {len(_out)} documents | teacher params "
          f"{_meta['params_total']/1e6:.1f}M | dev select-score {_meta['dev_select']}")
    print("caveat: on the TRAIN documents the teacher is overconfident (it was trained on them); its 'dev' distributions are out-of-sample.")
else:
    print("soft-label export skipped (EXPORT_SOFT_LABELS=False or smoke run)")

## What to do next

1. **Smoke test first:** `SMOKE_TEST=True` (a few minutes). Read the scaling table (LoRA's one-step update shrinks like 1/r, rsLoRA's stays flat), the *attention:* line (blocks 0–12 bidirectional), the *identity-at-init* line, the trainable-parameter count (about 8.1M for rsLoRA r = 64) and the smoke verdict.
2. **Full run:** `SMOKE_TEST=False`, defaults. It selects the epoch on dev (mean of the CL and IT macro-F1) and **already writes a valid `submission.zip`**: upload it to Codabench right away; everything below is refinement.
3. **Controls for the report** (change Cell 2 only, new `RUN_NAME` each time):
   * **rsLoRA vs LoRA on the same backbone:** `METHOD="lora"` (r = 16, α = 32) and, to see the high-rank problem itself, `METHOD="lora"` with `HP["lora"]["r"]=64, alpha=32` (γ = 0.5, the *unstabilised* high-rank LoRA) against the default rsLoRA r = 64. Only the scaling differs between the last two;
   * **PEFT vs full adaptation:** `METHOD="full"` and `METHOD="partial"` with `ATTN_MODE="bidir_top"`;
   * **attention:** `ATTN_MODE="causal"` (does unmasking help?) and `"bidir_top"`;
   * **context:** `CONTEXT_MODE="sentence"` and a smaller `WINDOW_TOKENS`.
4. **Final fit (optional):** `FINAL_FIT=True`, `EPOCHS=<best epoch>`, `BIAS_FINAL=<list printed by Cell 22>`.
5. **KD teacher:** at ≈62M parameters with the adapters this run is a valid teacher for the 36.6M student; set `EXPORT_SOFT_LABELS=True` if it is your best dev run.
6. Copy every `run_log.json` into the report folder (parameters, peak GPU memory, training time, per-label P/R/F1, confusions for CL and IT).